# 02 · Dense retrieval over 8.8M passages (GPU)

**Settings:** Accelerator **GPU T4 x2** (one is used) · Internet **On** · **Add Input → your notebook 01's output** · Save & Run All.

1. Encodes all 8.8M passages with `msmarco-MiniLM-L6-cos-v5` (fp16 on the GPU), in resumable shards.
2. Exact dense retrieval on the GPU (whole corpus in GPU memory, one matmul per query batch) for dev / DL19 / DL20.
3. Builds two IVF-PQ indexes (48 and 96 bytes per vector) for serving on the laptop, each with ANN recall measured against exact search.

Output ≈ 9 GB. Estimated runtime: 1.5–2 h.

In [ ]:
# Clone the project (outside /kaggle/working: code is not an output) and install it.
import os, subprocess, sys, time, json, glob, shutil
T0 = time.time()
REPO = os.environ.get("REPO", "https://github.com/Rishabh763/Search-engine-Neural-Ranking.git")
import socket
try:
    socket.gethostbyname("github.com")
except OSError:
    raise RuntimeError(
        "No internet in this notebook. In the right sidebar open Session options "
        "(or Settings) and switch Internet ON, then restart and run again. "
        "The toggle only appears once your Kaggle account is phone-verified "
        "(Settings > Phone verification).") from None
if not os.path.exists("/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/tmp/repo"], check=True)
if not os.path.exists("/tmp/repo/scripts/train_reranker.py"):
    raise RuntimeError(
        "The repo on GitHub is older than this notebook (no scripts/train_reranker.py). "
        "Commit and push from your laptop, then delete /tmp/repo (!rm -rf /tmp/repo) and rerun.")
os.chdir("/tmp/repo")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -e ".[dense]" 2>&1 | tail -3
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
def find_input(pattern):
    """Locate a file inside attached inputs (Kaggle mounts them under /kaggle/input/...)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no input matches {pattern}: attach the previous notebook's output (Add Input)")
    return hits[0]

NB01 = find_input("data/processed/msmarco-full/manifest.json").split("/data/processed/")[0]
print("notebook 01 output:", NB01)
os.environ.update(DATA_DIR=f"{NB01}/data/processed", INDEX_DIR="/kaggle/working/indexes",
                  RUNS_DIR="/kaggle/working/runs", PYTHONUTF8="1")
# Small things are copied so this notebook's output is self-contained; the 3 GB corpus stays in the input.
shutil.copytree(f"{NB01}/indexes/msmarco-full/docstore", "/kaggle/working/indexes/msmarco-full/docstore", dirs_exist_ok=True)
shutil.copytree(f"{NB01}/runs", "/kaggle/working/runs", dirs_exist_ok=True)
!python -c "from searchengine.envinfo import environment_info, short_label; print(short_label(environment_info()))"

## 1. Encode 8.8M passages (fp16, length-sorted batches; re-running skips finished shards)

In [ ]:
!python scripts/encode_corpus.py --collection msmarco-full --batch-size 512 --shard-size 500000 2>&1 | grep -v "Batches"

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 2. Exact dense retrieval on the GPU (ground truth for the ANN indexes)

In [ ]:
!python scripts/evaluate.py --collection msmarco-full --retriever dense-exact --latency-queries 0

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)

## 3. IVF-PQ indexes for the laptop (the full-precision vectors are 13.5 GB as fp32: too big for laptop RAM)

In [ ]:
!python scripts/build_dense_index.py --collection msmarco-full --type ivfpq --nlist 8192 --pq-m 48 --nprobe 64 --name ivfpq48

In [ ]:
!python scripts/build_dense_index.py --collection msmarco-full --type ivfpq --nlist 8192 --pq-m 96 --nprobe 64 --name ivfpq96

In [ ]:
for f in sorted(glob.glob("/kaggle/working/runs/msmarco-full/*/*.json")):
    r = json.load(open(f)); print(f"{r['eval_set']:12s} {r['tag']:45s} " + "  ".join(f"{k}={v:.4f}" for k, v in r["metrics"].items()))
for f in sorted(glob.glob("/kaggle/working/indexes/msmarco-full/dense/*/ivfpq*.json")):
    j = json.load(open(f)); print(os.path.basename(f), {k: j.get(k) for k in ("file_mb", "recall@10_vs_exact", "recall@100_vs_exact", "search_ms_per_query", "build_seconds")})

In [ ]:
print(f"elapsed: {(time.time() - T0) / 60:.1f} min"); print(subprocess.run(["du", "-sh", "/kaggle/working"], capture_output=True, text=True).stdout)